In [26]:
!pip install playwright

In [27]:
!playwright install

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Missing libraries:                                   ║
║     libwoff2dec.so.1.0.2                             ║
║     libgstgl-1.0.so.0                                ║
║     libgstcodecparsers-1.0.so.0                      ║
║     libavif.so.13                                    ║
║     libharfbuzz-icu.so.0                             ║
║     libenchant-2.so.2                                ║
║     libsecret-1.so.0                                 ║
║     libhyphen.so.0                                   ║
║     libmanette-0.2.so.0                              ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.11/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:269:9)
    at process.processTicksAndRejections (node:internal/process/task_queues:105

In [28]:
!pip install openai

In [29]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [30]:
#to test if the API key is working

from openai import OpenAI

client = OpenAI()
prompt = "hello, who are you?"
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[ {"role": "user", "content": prompt}]
)
response.choices[0].message.content

"Hello! I'm an AI language model created by OpenAI. I'm here to assist you with a wide range of questions and topics. How can I help you today?"

In [31]:
!pip install gradio

In [32]:
import gradio as gr

In [33]:
import os
from google.colab import userdata
os.environ['SERPER_API_KEY'] = userdata.get('SERPER_API_KEY')

In [34]:
!pip install requests

In [35]:
import requests
import os

#Serper API key and the search query
api_key = os.environ.get("SERPER_API_KEY")
query = "latest developments in generative AI"

#URL
url = "https://google.serper.dev/search"


headers = {
    'X-API-KEY': api_key,
    'Content-Type': 'application/json'
}


payload = {
    "q": query,
    "num": 3  # Number of search results
}

# Send a POST request to Serper API
response = requests.post(url, headers=headers, json=payload)

# Check request
response.raise_for_status()


data = response.json()


search_results = []
if 'organic' in data:
    for item in data['organic'][:3]: # Take top 3 organic results
        search_results.append({
            'title': item.get('title'),
            'url': item.get('link')
        })




print("Top search results:")
if search_results:
    for result in search_results:
        print(f"- {result.get('title')} ({result.get('url')})")
else:
    print("No organic search results found.")

Top search results:
- Generative AI news and analysis - TechCrunch (https://techcrunch.com/tag/generative-ai/)
- 5 Generative AI Trends To Watch Out For In 2025 - Forbes (https://www.forbes.com/sites/janakirammsv/2025/01/12/5-generative-ai-trends-to-watch-out-for-in-2025/)
- Top 10 Generative AI Trends in 2025 - SoluLab (https://www.solulab.com/top-generative-ai-trends/)


In [36]:
import asyncio
from playwright.async_api import async_playwright
import openai
from openai import OpenAI
import getpass

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

async def search_and_summarize(urls):
    async with async_playwright() as p:
        # Launch a Chromium browser
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        all_text_content = ""
        print("Scraping content from provided URLs...")
        for url_info in urls:
            url = url_info.get('url')
            if url:
                try:
                    await page.goto(url, timeout=30000)
                    content_elements = await page.locator('body p').all()

                    for element in content_elements:
                        text = await element.inner_text()
                        all_text_content += text + "\n"
                    print(f"Successfully scraped: {url}")
                except Exception as e:
                    print(f"Could not scrape {url}: {e}")

        if not all_text_content:
            await browser.close()
            return "Could not find sufficient content to summarize from the provided URLs."

        #Using OpenAI to summarize the content
        print("\nSummarizing content...")

        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that summarizes web page content."},
                {"role": "user", "content": f"Based on the following text from several web pages, provide a concise summary of the key findings in no more than 250 words:\n\n{all_text_content}"}
            ]
        )

        summary = completion.choices[0].message.content
        await browser.close()
        return summary


In [37]:
# call the async function
final_summary = await search_and_summarize(search_results)

print("\nFinal Summary:")
print(final_summary)

Scraping content from provided URLs...
Successfully scraped: https://techcrunch.com/tag/generative-ai/
Successfully scraped: https://www.forbes.com/sites/janakirammsv/2025/01/12/5-generative-ai-trends-to-watch-out-for-in-2025/
Successfully scraped: https://www.solulab.com/top-generative-ai-trends/

Summarizing content...

Final Summary:
In 2025, generative AI (GenAI) is set to radically transform enterprise applications, shifting from supplemental features to being integral components in software design. This evolution encompasses several key trends:

1. **AI-First Application Development**: Enterprises are increasingly integrating AI as a core element in software, enhancing workflow automation and intelligence through large language models (LLMs).

2. **Service as Software**: AI agents are evolving to automate tasks previously reliant on human intervention, changing the SaaS landscape towards an outcome-based pricing model where customers only pay for tasks completed by AI.

3. **Enha

In [38]:
import gradio as gr
import requests
import os
import asyncio

async def get_urls_from_serper(query):
    """Fetches search results URLs from Serper API."""
    api_key = os.environ.get("SERPER_API_KEY")
    url = "https://google.serper.dev/search"
    headers = {
        'X-API-KEY': api_key,
        'Content-Type': 'application/json'
    }
    payload = {
        "q": query,
        "num": 3 # Number of search results
    }
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()
        search_results = []
        if 'organic' in data:
            for item in data['organic'][:3]:
                search_results.append({'title': item.get('title'), 'url': item.get('link')})
        return search_results
    except Exception as e:
        print(f"Error fetching search results from Serper API: {e}")
        return []

async def summarize_search_results(query):
    """Performs search, scrapes content, and summarizes using OpenAI."""
    urls = await get_urls_from_serper(query)
    if not urls:
        return "Could not retrieve search results from Serper API."

    summary = await search_and_summarize(urls)
    return summary


#Gradio interface
iface = gr.Interface(
    fn=lambda query: asyncio.run(summarize_search_results(query)),
    inputs=gr.Textbox(label="Enter your search query"),
    outputs=gr.Textbox(label="Summary"),
    title="Serper API Search and Summarization",
    description="Enter a search query to get a summary of the top search results using Serper API and OpenAI."
)


iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbfc75aa6d45509e0e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
